# 🎮 Minecraft Server — v11 (EK CELL + advanced panel)

**Ab sirf EK kaam hai: neeche wale CELL ko ek baar Run karo — poora setup khud shuru ho jayega.**
Koi cell-to-cell jhanjhat nahi, koi overlap nahi — sab kuch sahi order me ek hi jagah hai.

## ▶️ Kaise chalana hai
1. ▶️ **Run** dabao (Run all bhi same cheez hai)
2. Google Drive ka **popup → ALLOW** (1 baar)
3. **Pehli baar** ek **box** aayega — usme **PLAYIT_SECRET paste** karo → Enter (Drive me save ho jayega, phir kabhi nahi poochenge)
4. Bas! **Panel link** 📱 ntfy notification + output me aa jayega — **tab khula rakho**, sab auto!

## 🆕 v11 naya kya hai
- **1 single cell** — ek baar Run = deps + Drive + secret + Java + jar + cloudflared + playit + **Panel v5** + agent + keepalive sab
- **Panel v5 (advanced!)**: live console, **live chat**, **CPU/RAM/TPS charts**, **file manager** (view/edit/delete/upload/download), **Modrinth se plugins** install/uninstall, **backups** (banao + download + restore), **software switcher** (Paper / Purpur / Vanilla / Fabric)
- **💾 Fast backups**: har **60 sec** save + Drive sync (pehle 5 min tha) + stop/restart/backup-button par turant
- **📱 Notifications**: panel link, server ONLINE, crash-restart, backup done — sab phone pe
- **Software switch**: panel me ek tap se Paper se Purpur/Vanilla/Fabric — jar khud download hoga

## 🛡️ Sab fallbacks jaise the
| Cheez | Primary | Fallback |
|---|---|---|
| **🌐 Panel link** | cloudflared | ntfy notification + notebook output |
| **☕ Java 25** | Adoptium JRE | JDK → Drive cache → Azul |
| **📦 server.jar** | Paper/Purpur/Vanilla/Fabric | Drive cache |
| **💾 World** | local save | Drive sync har 60 sec + zip backups |
| **🤖 Server/panel/tunnels** | auto-restart watchdog (30 sec) | — |

**⚠️ ZAROORI:** Ye tab khula rakho • Colab limit ~12h ke baad wapas **wahi cell dobara Run** (world Drive se wapas aa jayega)

In [ ]:
# ══════════════════════════════════════════════════════════
# 🎮 MC SERVER v11 — BAS YE EK CELL RUN KARO (sab kuch isi me!)
# ══════════════════════════════════════════════════════════
import os, sys, time, json, subprocess, threading, re, hashlib, shutil
import urllib.request
from datetime import datetime

print('=' * 60)
print('🚀 MC Server v11 — single-cell boot shuru')
print('=' * 60)

!sudo apt-get update -qq && sudo apt-get install -y -qq screen wget curl rsync procps psmisc
!pip install -q flask psutil
print('✅ Base deps installed!')

# ── zip/unzip (backups ke liye) ──
subprocess.run('sudo apt-get install -y -qq zip unzip', shell=True, capture_output=True)
print('✅ zip/unzip ready')

# 🧹 AUTO-CLEANUP — is session ke purane sab processes band (re-run safe)
import subprocess, time, os

subprocess.run("pkill -f 'cloudflared' 2>/dev/null || true", shell=True, capture_output=True)
subprocess.run("pkill -f 'panel.py' 2>/dev/null || true", shell=True, capture_output=True)
subprocess.run("pkill -x playit 2>/dev/null || true", shell=True, capture_output=True)
subprocess.run("pkill -f 'server.jar' 2>/dev/null || true", shell=True, capture_output=True)
subprocess.run("fuser -k 8081/tcp >/dev/null 2>&1 || true", shell=True)
subprocess.run("fuser -k 4040/tcp >/dev/null 2>&1 || true", shell=True)
subprocess.run(['screen', '-S', 'mcserver', '-X', 'quit'], capture_output=True)
subprocess.run(['screen', '-wipe'], capture_output=True)
for f in ('STOP_FLAG', 'CF_URL', 'START_LOCK', 'BACKUP_REQUEST', 'RESTORE_REQUEST', 'SOFTWARE_STATUS'):
    if os.path.exists(f):
        os.remove(f)
time.sleep(2)
print('✅ Cleanup done — clean start!')

# 💾 GOOGLE DRIVE — world backup + Java/jar cache
import os, subprocess

DRIVE_BASE = None
try:
    if os.path.exists('/content/drive/MyDrive'):
        print('💾 Drive pehle se connected hai')
    else:
        from google.colab import drive
        print('⏳ Google Drive connect ho raha hai...')
        print('   👉 POPUP aaye to apna account choose karke ALLOW karo (sirf 1 baar)')
        drive.mount('/content/drive')
    base = '/content/drive/MyDrive/MC_Colab'
    os.makedirs(base, exist_ok=True)
    DRIVE_BASE = base
    print('💾 Drive connected:', base, '(world + caches yahan save honge)')
except Exception as e:
    print('⚠️ Drive connect nahi hua:', e)
    print('   ℹ️ Koi baat nahi — sab chalega! Bas world sirf local rahega aur cache nahi milega.')

# Purana world Drive me hai to wapas laao (naye session me)
if DRIVE_BASE:
    for d in ('world', 'world_nether', 'world_the_end'):
        src = os.path.join(DRIVE_BASE, d)
        dst = '/content/' + d
        if os.path.isdir(src):
            if not os.path.isdir(dst):
                subprocess.run(['rsync', '-a', src + '/', dst + '/'], capture_output=True)
                print('✅ Wapas aaya (Drive se):', d)
            else:
                print('ℹ️ Local me pehle se hai:', d)
    print('💾 World backup ON — har 60 sec Drive me sync hoga')
else:
    print('ℹ️ World backup OFF (sirf is session ke liye chalega)')

# ── STAGE 3b: configs + software stamp Drive se wapas (naye session me) ──
if DRIVE_BASE:
    for _f in ('server.properties', 'ops.json', 'whitelist.json', 'banned-ips.json', 'banned-players.json', 'SOFTWARE', 'MC_VERSION'):
        _src = os.path.join(DRIVE_BASE, _f)
        if os.path.exists(_src) and not os.path.exists(_f):
            try:
                shutil.copy(_src, _f)
            except Exception:
                pass
    if os.path.isdir(os.path.join(DRIVE_BASE, 'plugins')) and not os.path.isdir('plugins'):
        try:
            subprocess.run(['rsync', '-a', '--exclude', '*.jar', os.path.join(DRIVE_BASE, 'plugins') + '/', 'plugins/'], capture_output=True)
        except Exception:
            pass
    print('💾 Configs + software stamp restore check done')

# 🔑 SECRET — bas EK BAAR! (paste → Drive me save → hamesha yaad)
# PLAYIT_SECRET tumhara Minecraft address chalata hai (chat me di hai)
# Isi se 2 cheezein khud ban jaati hain: 📱 ntfy topic + 🔒 panel key
import os, json, hashlib

SEC_FILE = os.path.join(DRIVE_BASE, 'secrets.json') if DRIVE_BASE else None
PLAYIT_SECRET = ''
NTFY_TOPIC = ''
PANEL_KEY = ''

# 1) Drive me pehle se save hai? (aam taur pe yahin milega)
try:
    if SEC_FILE and os.path.exists(SEC_FILE):
        PLAYIT_SECRET = (json.load(open(SEC_FILE)) or {}).get('playit_secret', '') or ''
        if PLAYIT_SECRET:
            print('✅ Secret Drive se mil gaya (hamesha yaad hai!)')
except Exception as e:
    print('ℹ️ Drive secret read skip:', str(e)[:80])

# 2) Colab Secrets me ho to wahan se (agar kabhi daala ho)
if not PLAYIT_SECRET:
    try:
        from google.colab import userdata
        PLAYIT_SECRET = userdata.get('PLAYIT_SECRET') or ''
        if PLAYIT_SECRET:
            print('✅ Secret Colab 🔑 se mil gaya')
    except Exception:
        pass

# 3) Pehli baar: box me paste karo → Drive me save
if not PLAYIT_SECRET:
    print('═' * 55)
    print('🔑 PEHLI BAAR: niche wale BOX me PLAYIT_SECRET paste karke Enter dabao')
    print('   (Value chat me di hai — "5a409bfc..." wali)')
    print('   Ye sirf 1 baar — phir Drive me hamesha ke liye save ho jayega!')
    print('═' * 55)
    try:
        PLAYIT_SECRET = input('PLAYIT_SECRET paste karo: ').strip()
    except Exception:
        PLAYIT_SECRET = ''
    if PLAYIT_SECRET and SEC_FILE:
        try:
            json.dump({'playit_secret': PLAYIT_SECRET}, open(SEC_FILE, 'w'))
            print('💾 Drive me save ho gaya — ab kabhi nahi poochenge!')
        except Exception as e:
            print('⚠️ Drive me save nahi hua:', str(e)[:80])

if PLAYIT_SECRET:
    NTFY_TOPIC = 'mc-yk-' + hashlib.sha256(('topic|' + PLAYIT_SECRET).encode()).hexdigest()[:24]
    PANEL_KEY = hashlib.sha256(('key|' + PLAYIT_SECRET).encode()).hexdigest()[:16]
    open('PANEL_KEY', 'w').write(PANEL_KEY)
    print('✅ Secret ready — Minecraft + 📱 notification + 🔒 panel lock sab ON')
    print('📱 Apna notification channel (isko bookmark karo):')
    print('   https://ntfy.sh/' + NTFY_TOPIC)
else:
    print('❌ Secret nahi mila — Minecraft nahi chalega (panel phir bhi chalega)')
    print('   Fix: ye cell dobara Run karo aur box me paste karo')
    import secrets as _pysecrets
    PANEL_KEY = _pysecrets.token_hex(8)
    open('PANEL_KEY', 'w').write(PANEL_KEY)
    print('🔒 Is session ka temporary panel key:', PANEL_KEY)

# ☕ JAVA 25 (Paper 26.3 ke liye ZAROORI) — 4-step fallback chain
import urllib.request, subprocess, os, re, shutil

UA = {'User-Agent': 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'}
JAVA = '/opt/java25/bin/java'

def java_ok():
    if not os.path.exists(JAVA):
        return False
    try:
        r = subprocess.run([JAVA, '-version'], capture_output=True, text=True, timeout=30)
        txt = (r.stderr or '') + (r.stdout or '')
        return '25.' in txt
    except Exception:
        return False

def fetch(url, dest):
    req = urllib.request.Request(url, headers=UA)
    with urllib.request.urlopen(req, timeout=300) as r, open(dest, 'wb') as f:
        got = 0
        while True:
            c = r.read(1024 * 1024)
            if not c:
                break
            f.write(c)
            got += len(c)
    return got

def extract(tarfile):
    subprocess.run('rm -rf /opt/java25 && mkdir -p /opt/java25', shell=True)
    subprocess.run('tar xzf ' + tarfile + ' -C /opt/java25 --strip-components=1', shell=True, check=True)
    return java_ok()

if java_ok():
    print('✅ Java 25 pehle se ready hai')
else:
    done = False

    # 1 + 2) Adoptium (official Temurin) — JRE phir JDK
    for name, url, fname in [
        ('Adoptium JRE', 'https://api.adoptium.net/v3/binary/latest/25/ga/linux/x64/jre/hotspot/normal/eclipse', 'jre25.tar.gz'),
        ('Adoptium JDK', 'https://api.adoptium.net/v3/binary/latest/25/ga/linux/x64/jdk/hotspot/normal/eclipse', 'jdk25.tar.gz'),
    ]:
        if done:
            break
        try:
            print('⬇️ Java 25 (' + name + ') download...')
            mb = fetch(url, fname) / 1024 / 1024
            print('   mila: %.0f MB — extract...' % mb)
            if extract(fname):
                print('✅ Java 25 ready (' + name + ')')
                done = True
        except Exception as e:
            print('⚠️ ' + name + ' fail:', str(e)[:120])

    # 3) Drive cache
    if not done:
        try:
            if DRIVE_BASE:
                cached = os.path.join(DRIVE_BASE, 'jre25.tar.gz')
                if os.path.exists(cached):
                    print('⬇️ Java Drive cache se aa raha hai...')
                    shutil.copy(cached, 'jre25.tar.gz')
                    if extract('jre25.tar.gz'):
                        print('✅ Java 25 ready (Drive cache)')
                        done = True
        except Exception as e:
            print('⚠️ Drive cache fail:', str(e)[:120])

    # 4) Azul Zulu
    if not done:
        try:
            print('⬇️ Azul (last fallback) try...')
            html = urllib.request.urlopen(urllib.request.Request('https://cdn.azul.com/zulu/bin/', headers=UA), timeout=120).read().decode('utf-8', 'ignore')
            cands = sorted(set(re.findall(r'zulu25\.[0-9.]+-ca-(?:jre|jdk)25\.[0-9.]+-linux_x64\.tar\.gz', html)))
            if cands:
                print('   mila:', cands[-1])
                fetch('https://cdn.azul.com/zulu/bin/' + cands[-1], 'zulu25.tar.gz')
                if extract('zulu25.tar.gz'):
                    print('✅ Java 25 ready (Azul)')
                    done = True
        except Exception as e:
            print('⚠️ Azul fail:', str(e)[:120])

    # Success hua to Drive me cache save karo (agle session ke liye)
    if done:
        try:
            if DRIVE_BASE:
                cached = os.path.join(DRIVE_BASE, 'jre25.tar.gz')
                if not os.path.exists(cached):
                    for f in ('jre25.tar.gz', 'zulu25.tar.gz'):
                        if os.path.exists(f):
                            shutil.copy(f, cached)
                            print('💾 Java cache Drive me save')
                            break
        except Exception as e:
            print('ℹ️ cache save skip:', str(e)[:80])
    else:
        print('❌ Java install fail — is cell ko dobara Run karo (internet issue)')

# java command ko v25 banao + verify
try:
    if java_ok():
        if os.path.lexists('/usr/local/bin/java'):
            os.remove('/usr/local/bin/java')
        os.symlink(JAVA, '/usr/local/bin/java')
        r = subprocess.run([JAVA, '-version'], capture_output=True, text=True)
        ver = ((r.stderr or r.stdout) or '').strip().splitlines()[0] if (r.stderr or r.stdout) else ''
        print('Java version:', ver)
        print('✅ Java 25 READY!')
except Exception as e:
    print('⚠️ symlink issue (chalega):', e)

# ── STAGE 6: server.jar (software-aware + Drive cache) ──
_UA = {'User-Agent': 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'}

def _jar_size():
    try:
        return os.path.getsize('server.jar')
    except Exception:
        return 0

SOFTWARE = ''
try:
    SOFTWARE = open('SOFTWARE').read().strip()
except Exception:
    pass
if SOFTWARE not in ('paper', 'purpur', 'vanilla', 'fabric'):
    SOFTWARE = 'paper'
try:
    if not os.path.exists('SOFTWARE'):
        open('SOFTWARE', 'w').write(SOFTWARE)
except Exception:
    pass
if not os.path.exists('MC_VERSION'):
    open('MC_VERSION', 'w').write('26.3')
_MIN = 20 * 1024 if SOFTWARE == 'fabric' else 40 * 1024 * 1024

def _paper_dl():
    try:
        req = urllib.request.Request('https://fill.papermc.io/v3/projects/paper/versions/26.3/builds', headers=_UA)
        latest = json.loads(urllib.request.urlopen(req, timeout=30).read())[0]
        print('📦 Paper 26.3 latest build: #' + str(latest['id']))
        dl = latest['downloads']['server:default']['url']
        req2 = urllib.request.Request(dl, headers=_UA)
        with urllib.request.urlopen(req2, timeout=300) as r, open('server.jar', 'wb') as f:
            got = 0
            while True:
                c = r.read(1024 * 1024)
                if not c:
                    break
                f.write(c)
                got += len(c)
        print('   download: %.0f MB' % (got / 1024 / 1024))
        return True
    except Exception as e:
        print('⚠️ Paper download fail:', str(e)[:140])
        return False

if _jar_size() < _MIN and SOFTWARE == 'paper':
    _paper_dl()

if _jar_size() < _MIN:
    try:
        if DRIVE_BASE:
            cached = os.path.join(DRIVE_BASE, 'server.jar')
            if os.path.exists(cached) and os.path.getsize(cached) >= _MIN:
                shutil.copy(cached, 'server.jar')
                print('✅ server.jar Drive cache se aa gaya')
    except Exception as e:
        print('⚠️ Drive cache fail:', str(e)[:120])

if _jar_size() < _MIN and SOFTWARE != 'paper':
    print('⚠️ ' + SOFTWARE + ' jar nahi mila — panel ke Software tab me "' + SOFTWARE + '" Apply karo')
    if _paper_dl():
        SOFTWARE = 'paper'
        open('SOFTWARE', 'w').write('paper')

if _jar_size() >= _MIN:
    try:
        if DRIVE_BASE:
            cached = os.path.join(DRIVE_BASE, 'server.jar')
            if (not os.path.exists(cached)) or os.path.getsize(cached) != _jar_size():
                shutil.copy('server.jar', cached)
                print('💾 server.jar Drive cache updated')
            open(os.path.join(DRIVE_BASE, 'SOFTWARE'), 'w').write(SOFTWARE)
    except Exception:
        pass
    print('✅ server.jar ready: %.1f MB (%s)' % (_jar_size() / 1024 / 1024, SOFTWARE))
else:
    print('❌ server.jar nahi mila — ye cell dobara Run karo (internet issue)')

open('eula.txt', 'w').write('eula=true\n')
if not os.path.exists('server.properties'):
    open('server.properties', 'w').write(
        'motd=Colab SMP - Dosto ka Adda\n'
        'max-players=10\n'
        'difficulty=normal\n'
        'gamemode=survival\n'
        'view-distance=8\n'
        'simulation-distance=6\n'
        'spawn-protection=0\n')
print('✅ eula.txt + server.properties ready')

# 🌐 cloudflared (PANEL LINK — koi account nahi, koi takrav nahi!)
import os, subprocess

if not os.path.exists('cloudflared') or os.path.getsize('cloudflared') < 5 * 1024 * 1024:
    print('⬇️ cloudflared download (backup tunnel, ~40 MB)...')
    subprocess.run("curl -sL --retry 3 --retry-delay 2 -f https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o cloudflared", shell=True)
    subprocess.run('chmod +x cloudflared', shell=True)

if os.path.exists('cloudflared') and os.path.getsize('cloudflared') > 5 * 1024 * 1024:
    print('✅ cloudflared ready (panel link)')
else:
    print('⚠️ cloudflared download fail — panel link nahi milega! Ye cell dobara Run karo')

# ── STAGE 8: Panel v5 likho ──
PANEL_CODE = r'''"""Minecraft Colab Panel v5.0 — modern dashboard: live console, chat, charts (CPU/RAM/TPS),
file manager (view/edit/delete/upload/download), plugins (Modrinth + URL + upload),
backups (create/download/restore), software switcher (Paper/Purpur/Vanilla/Fabric).
Key-protected (PANEL_KEY). No triple-single-quote sequences anywhere (notebook embeds this)."""
from flask import Flask, jsonify, request, redirect, send_file
import subprocess, os, threading, time, hmac, json, re, shutil, urllib.request, urllib.parse
from datetime import datetime

app = Flask(__name__)
app.config['MAX_CONTENT_LENGTH'] = 300 * 1024 * 1024

BASE = os.getcwd()
TMP = os.path.join(BASE, '_panel_tmp')
DRIVE_MC = (os.environ.get('DRIVE_MC_DIR') or '/content/drive/MyDrive/MC_Colab').strip()
DRIVE_BACKUPS = os.path.join(DRIVE_MC, 'backups')
JAVA = '/opt/java25/bin/java'
FLAG = 'STOP_FLAG'
UA = 'MC-Colab-Panel/5.0 (+https://github.com/ykewat354-cyber/mc-colab)'

KEY = (os.environ.get('PANEL_KEY') or '').strip()
if not KEY:
    try:
        KEY = open(os.path.join(BASE, 'PANEL_KEY')).read().strip()
    except Exception:
        KEY = ''


def _key_ok(val):
    try:
        return bool(val) and bool(KEY) and hmac.compare_digest(str(val), str(KEY))
    except Exception:
        return False


LOCK_PAGE = """<!DOCTYPE html>
<html lang="en"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Locked</title>
<style>body{font-family:system-ui,sans-serif;background:#0b1220;color:#e2e8f0;margin:0;padding:40px 16px;text-align:center}
.c{max-width:420px;margin:0 auto;background:#121b2e;border-radius:16px;padding:28px 20px}
.muted{color:#94a3b8;font-size:13px;line-height:1.7}</style></head>
<body><div class="c">
<h2>Panel Locked</h2>
<p class="muted">Ye panel sirf key-wale link se khulta hai.<br>
Wahi link <b>ntfy notification</b> aur <b>notebook ke output</b> me milta hai &mdash; usko kholo, phir panel hamesha khulega.</p>
</div></body></html>"""


@app.before_request
def _gate():
    if not KEY:
        return
    if request.path == '/favicon.ico':
        return
    if _key_ok(request.args.get('key', '')):
        resp = redirect(request.path)
        resp.set_cookie('mcpanel', KEY, max_age=31536000, samesite='Lax')
        return resp
    if _key_ok(request.cookies.get('mcpanel', '')):
        return
    if request.path.startswith('/api/'):
        return jsonify({'error': 'locked'}), 401
    return LOCK_PAGE, 401


@app.after_request
def _nocache(resp):
    if request.path.startswith('/api/'):
        resp.headers['Cache-Control'] = 'no-store'
    return resp


@app.errorhandler(413)
def _too_big(e):
    return jsonify({'error': 'file bahut badi hai (max 300MB)'}), 413


def read_file(name, default=''):
    try:
        return open(os.path.join(BASE, name), errors='ignore').read().strip()
    except Exception:
        return default


def write_file(name, text):
    p = os.path.join(BASE, name)
    tmp = p + '.tmp'
    with open(tmp, 'w', encoding='utf-8') as f:
        f.write(text)
    os.replace(tmp, p)


def tail_bytes(path, n):
    try:
        p = path if os.path.isabs(path) else os.path.join(BASE, path)
        with open(p, 'rb') as f:
            f.seek(0, 2)
            size = f.tell()
            f.seek(max(0, size - n))
            return f.read().decode('utf-8', 'ignore')
    except Exception:
        return ''


def _u(pattern, exact=False):
    cmd = ['pgrep', '-x', pattern] if exact else ['pgrep', '-f', pattern]
    try:
        return subprocess.run(cmd, capture_output=True).returncode == 0
    except Exception:
        return False


def is_running():
    try:
        return 'mcserver' in subprocess.run(['screen', '-ls'], capture_output=True, text=True).stdout
    except Exception:
        return False


def stuff(cmd):
    subprocess.run(['screen', '-S', 'mcserver', '-X', 'stuff', cmd + '\n'], capture_output=True)


def _start_allowed():
    try:
        if os.path.exists('START_LOCK'):
            if time.time() - os.path.getmtime('START_LOCK') < 120:
                return False
    except Exception:
        pass
    try:
        open('START_LOCK', 'w').write(str(time.time()))
    except Exception:
        pass
    return True


def do_start():
    if is_running():
        if os.path.exists(FLAG):
            os.remove(FLAG)
        return
    if not _start_allowed():
        return
    subprocess.run(['screen', '-S', 'mcserver', '-X', 'quit'], capture_output=True)
    subprocess.run(['screen', '-wipe'], capture_output=True)
    subprocess.Popen(['screen', '-dmS', 'mcserver', 'bash', '-c',
                      JAVA + ' -Xmx2G -Xms2G -jar server.jar nogui 2>&1 | tee server_console.log'])
    try:
        write_file('SERVER_STARTED_AT', str(time.time()))
    except Exception:
        pass
    if os.path.exists(FLAG):
        os.remove(FLAG)


def do_stop():
    open(os.path.join(BASE, FLAG), 'w').write('1')
    stuff('save-all flush')
    time.sleep(1)
    stuff('stop')


def do_restart():
    open(os.path.join(BASE, FLAG), 'w').write('1')
    stuff('save-all flush')
    time.sleep(1)
    stuff('stop')
    for _ in range(25):
        time.sleep(3)
        if not is_running():
            break
    subprocess.run(['screen', '-S', 'mcserver', '-X', 'quit'], capture_output=True)
    subprocess.run(['screen', '-wipe'], capture_output=True)
    if os.path.exists(FLAG):
        os.remove(FLAG)
    do_start()


# ---------------- stats sampler ----------------
HIST = {'t': [], 'cpu': [], 'ram': [], 'tps': [], 'players': []}
LATEST = {'cpu': 0.0, 'ram_mb': 0, 'sys_ram': 0.0, 'tps': None, 'mspt': None}
PLAYERS = {'count': 0, 'max': 0, 'names': []}


def _cpu_times():
    try:
        with open('/proc/stat') as f:
            parts = f.readline().split()
        vals = [int(x) for x in parts[1:8]]
        idle = vals[3] + vals[4]
        return sum(vals), idle
    except Exception:
        return 0, 0


def _mc_ram_mb():
    total = 0
    try:
        r = subprocess.run(['pgrep', '-f', 'server.jar'], capture_output=True, text=True)
        for pid in (r.stdout or '').split()[:3]:
            try:
                with open('/proc/' + pid + '/status') as f:
                    for ln in f:
                        if ln.startswith('VmRSS:'):
                            total += int(ln.split()[1]) // 1024
                            break
            except Exception:
                pass
    except Exception:
        pass
    return total


def _sys_ram_pct():
    try:
        info = {}
        with open('/proc/meminfo') as f:
            for ln in f:
                k, v = ln.split(':', 1)
                info[k] = int(v.strip().split()[0])
        return round(100.0 * (1 - info['MemAvailable'] / info['MemTotal']), 1)
    except Exception:
        return 0.0


def _refresh_parse():
    txt = tail_bytes('server_console.log', 30000)
    tps = re.findall(r'TPS from last 1m, 5m, 15m: ([0-9.]+)', txt)
    msp = re.findall(r'MSPT from last 1m, 5m, 15m: ([0-9.]+)', txt)
    if tps:
        LATEST['tps'] = float(tps[-1])
    if msp:
        LATEST['mspt'] = float(msp[-1])
    pl = re.findall(r'There are (\d+) of a max of (\d+) players online[:\s]*(.*)', txt)
    if pl:
        c, m, names = pl[-1]
        PLAYERS['count'] = int(c)
        PLAYERS['max'] = int(m)
        PLAYERS['names'] = [x.strip() for x in names.split(',') if x.strip()]


def sampler():
    pt, pi = _cpu_times()
    while True:
        time.sleep(5)
        try:
            t, i = _cpu_times()
            dt = t - pt
            di = i - pi
            pt, pi = t, i
            cpu = round(100.0 * (1 - di / dt), 1) if dt > 0 else 0.0
            LATEST['cpu'] = cpu
            LATEST['ram_mb'] = _mc_ram_mb()
            LATEST['sys_ram'] = _sys_ram_pct()
            _refresh_parse()
            HIST['t'].append(datetime.now().strftime('%H:%M:%S'))
            HIST['cpu'].append(cpu)
            HIST['ram'].append(LATEST['ram_mb'])
            HIST['tps'].append(LATEST['tps'] if LATEST['tps'] is not None else 0)
            HIST['players'].append(PLAYERS['count'])
            for k in ('t', 'cpu', 'ram', 'tps', 'players'):
                if len(HIST[k]) > 360:
                    HIST[k] = HIST[k][-360:]
        except Exception:
            pass


# ---------------- chat parse ----------------
def parse_chat(limit=120, scan=200000):
    txt = tail_bytes('server_console.log', scan)
    items = []
    rx_chat = re.compile(r'^\[(\d{2}:\d{2}:\d{2})\] \[[^\]]+\]: <([^>]+)> (.*)$')
    rx_join = re.compile(r'^\[(\d{2}:\d{2}:\d{2})\] \[[^\]]+\]: ([A-Za-z0-9_]{1,16}) joined the game$')
    rx_leave = re.compile(r'^\[(\d{2}:\d{2}:\d{2})\] \[[^\]]+\]: ([A-Za-z0-9_]{1,16}) left the game$')
    rx_adv = re.compile(r'^\[(\d{2}:\d{2}:\d{2})\] \[[^\]]+\]: ([A-Za-z0-9_]{1,16}) has made the advancement \[(.+)\]$')
    rx_say = re.compile(r'^\[(\d{2}:\d{2}:\d{2})\] \[[^\]]+\]: \[Server\] (.*)$')
    for ln in txt.split('\n'):
        m = rx_chat.match(ln)
        if m:
            items.append({'time': m.group(1), 'kind': 'chat', 'user': m.group(2), 'text': m.group(3)[:300]})
            continue
        m = rx_join.match(ln)
        if m:
            items.append({'time': m.group(1), 'kind': 'join', 'user': m.group(2), 'text': 'joined the game'})
            continue
        m = rx_leave.match(ln)
        if m:
            items.append({'time': m.group(1), 'kind': 'leave', 'user': m.group(2), 'text': 'left the game'})
            continue
        m = rx_adv.match(ln)
        if m:
            items.append({'time': m.group(1), 'kind': 'adv', 'user': m.group(2), 'text': 'made the advancement [' + m.group(3)[:80] + ']'})
            continue
        m = rx_say.match(ln)
        if m:
            items.append({'time': m.group(1), 'kind': 'server', 'user': 'Server', 'text': m.group(2)[:300]})
    return items[-limit:]


# ---------------- safe paths / files ----------------
def safe_path(rel):
    rel = (rel or '').strip().lstrip('/')
    p = os.path.realpath(os.path.join(BASE, rel))
    if p != BASE and not p.startswith(BASE + os.sep):
        raise ValueError('bad path')
    return p


def rel_of(p):
    r = os.path.relpath(p, BASE)
    return '' if r == '.' else r


FILE_SKIP = {'_panel_tmp'}


def list_dir(rel):
    target = safe_path(rel)
    if not os.path.isdir(target):
        raise ValueError('folder nahi mila')
    entries = []
    for e in sorted(os.scandir(target), key=lambda x: (not x.is_dir(), x.name.lower())):
        if e.name in FILE_SKIP and rel == '':
            continue
        try:
            st = e.stat()
            entries.append({'name': e.name, 'dir': e.is_dir(), 'size': (st.st_size if e.is_file() else 0),
                            'mtime': datetime.fromtimestamp(st.st_mtime).strftime('%d %b %H:%M')})
        except Exception:
            entries.append({'name': e.name, 'dir': False, 'size': 0, 'mtime': '-'})
    parent = rel_of(os.path.dirname(target)) if rel else None
    return {'path': rel, 'parent': parent, 'entries': entries}


# ---------------- software (server type) ----------------
SOFT_INFO = {
    'paper': {'label': 'Paper', 'desc': 'Best overall - plugins + performance (recommended)'},
    'purpur': {'label': 'Purpur', 'desc': 'Paper-based, extra customization options'},
    'vanilla': {'label': 'Vanilla (Mojang)', 'desc': 'Original Minecraft - no plugins'},
    'fabric': {'label': 'Fabric', 'desc': 'For mods - bukkit plugins not supported'},
}


def http_get(url, timeout=30):
    req = urllib.request.Request(url, headers={'User-Agent': UA})
    return urllib.request.urlopen(req, timeout=timeout)


def resolve_software(pid):
    mcv = read_file('MC_VERSION', '26.3') or '26.3'
    if pid == 'paper':
        d = json.loads(http_get('https://fill.papermc.io/v3/projects/paper/versions/' + mcv + '/builds').read())
        latest = d[0]
        return latest['downloads']['server:default']['url'], 5 * 1024 * 1024, 'paper ' + mcv
    if pid == 'purpur':
        d = json.loads(http_get('https://api.purpurmc.org/v2/purpur').read())
        vers = d.get('versions', [])
        ver = mcv if mcv in vers else (vers[-1] if vers else mcv)
        return 'https://api.purpurmc.org/v2/purpur/' + ver + '/latest/download', 5 * 1024 * 1024, 'purpur ' + ver
    if pid == 'vanilla':
        d = json.loads(http_get('https://launchermeta.mojang.com/mc/game/version_manifest_v2.json').read())
        pick = None
        for v in d.get('versions', []):
            if v.get('id') == mcv:
                pick = v
                break
        if not pick:
            lid = d.get('latest', {}).get('release')
            for v in d.get('versions', []):
                if v.get('id') == lid:
                    pick = v
                    break
        if not pick:
            raise RuntimeError('version nahi mila')
        vd = json.loads(http_get(pick['url']).read())
        return vd['downloads']['server']['url'], 5 * 1024 * 1024, 'vanilla ' + str(pick.get('id', mcv))
    if pid == 'fabric':
        loaders = json.loads(http_get('https://meta.fabricmc.net/v2/versions/loader/' + mcv).read())
        if not loaders:
            raise RuntimeError('Fabric ' + mcv + ' ke liye available nahi hai')
        loader = loaders[0]['loader']['version']
        inst = json.loads(http_get('https://meta.fabricmc.net/v2/versions/installer').read())
        installer = inst[0]['version']
        for i in inst:
            if i.get('stable'):
                installer = i['version']
                break
        url = 'https://meta.fabricmc.net/v2/versions/loader/' + mcv + '/' + loader + '/' + installer + '/server/jar'
        return url, 20 * 1024, 'fabric ' + mcv
    raise RuntimeError('unknown software')


def download_software(pid, dest, status_name='SOFTWARE_STATUS'):
    url, min_size, tag = resolve_software(pid)
    tmp = dest + '.new'
    got = 0
    write_file(status_name, 'download shuru: ' + tag)
    with http_get(url, timeout=300) as r, open(tmp, 'wb') as f:
        while True:
            c = r.read(1024 * 1024)
            if not c:
                break
            f.write(c)
            got += len(c)
            if got % (10 * 1024 * 1024) < 1024 * 1024:
                write_file(status_name, tag + ' download: ' + str(got // 1048576) + ' MB...')
    if got < min_size:
        try:
            os.remove(tmp)
        except Exception:
            pass
        raise RuntimeError('file chhoti mili (' + str(got) + ' bytes)')
    os.replace(tmp, dest)
    return got, tag


def software_switch(pid):
    def job():
        try:
            write_file('SOFTWARE_STATUS', 'server band kar rahe hain (safe save ke saath)...')
            open(os.path.join(BASE, FLAG), 'w').write('1')
            if is_running():
                stuff('save-all flush')
                time.sleep(2)
                stuff('stop')
                for _ in range(25):
                    time.sleep(3)
                    if not is_running():
                        break
            subprocess.run(['screen', '-S', 'mcserver', '-X', 'quit'], capture_output=True)
            subprocess.run(['screen', '-wipe'], capture_output=True)
            got, tag = download_software(pid, os.path.join(BASE, 'server.jar'))
            write_file('MC_VERSION', tag.split(' ')[-1])
            write_file('SOFTWARE', pid)
            write_file('SOFTWARE_STATUS', pid + ' ready (' + str(got // 1048576) + ' MB) - server start ho raha hai...')
        except Exception as e:
            write_file('SOFTWARE_STATUS', 'FAIL: ' + str(e)[:180])
        finally:
            if os.path.exists(FLAG):
                os.remove(FLAG)
    threading.Thread(target=job, daemon=True).start()


# ---------------- plugins ----------------
def plugins_dir():
    d = os.path.join(BASE, 'plugins')
    os.makedirs(d, exist_ok=True)
    return d


def _plug_name(name):
    return re.sub(r'[^A-Za-z0-9._-]', '_', os.path.basename(name or 'plugin.jar'))


def plugin_list():
    d = plugins_dir()
    out = []
    for n in sorted(os.listdir(d)):
        if n.endswith('.jar') or n.endswith('.jar.disabled'):
            p = os.path.join(d, n)
            st = os.stat(p)
            out.append({'name': n, 'enabled': n.endswith('.jar'),
                        'size': st.st_size, 'mtime': datetime.fromtimestamp(st.st_mtime).strftime('%d %b %H:%M')})
    return out


def download_to(url, dest):
    tmp = dest + '.new'
    got = 0
    with http_get(url, timeout=180) as r, open(tmp, 'wb') as f:
        while True:
            c = r.read(1024 * 1024)
            if not c:
                break
            f.write(c)
            got += len(c)
    if got < 2000:
        try:
            os.remove(tmp)
        except Exception:
            pass
        raise RuntimeError('file chhoti mili')
    os.replace(tmp, dest)
    return got


def modrinth_install(project):
    mcv = read_file('MC_VERSION', '26.3') or '26.3'
    q = urllib.parse.quote
    base = 'https://api.modrinth.com/v2/project/' + q(project, safe='') + '/version'
    attempts = [base + '?loaders=' + q('["paper"]') + '&game_versions=' + q('["' + mcv + '"]'),
                base + '?loaders=' + q('["paper"]'),
                base + '?loaders=' + q('["bukkit"]') + '&game_versions=' + q('["' + mcv + '"]'),
                base]
    vers = []
    for u in attempts:
        try:
            vers = json.loads(http_get(u).read())
            if vers:
                break
        except Exception:
            continue
    if not vers:
        raise RuntimeError('Modrinth pe koi version nahi mila')
    v = vers[0]
    jar = None
    for f in v.get('files', []):
        if f.get('filename', '').endswith('.jar') and f.get('primary'):
            jar = f
            break
    if not jar:
        for f in v.get('files', []):
            if f.get('filename', '').endswith('.jar'):
                jar = f
                break
    if not jar:
        raise RuntimeError('jar file nahi mili')
    dest = os.path.join(plugins_dir(), _plug_name(jar['filename']))
    got = download_to(jar['url'], dest)
    return os.path.basename(dest), got


def modrinth_search(q):
    qq = urllib.parse.quote
    base = 'https://api.modrinth.com/v2/search?query=' + qq(q) + '&limit=12&index=downloads'
    tries = [base + '&facets=' + qq('[["project_type:plugin"],["categories:paper"]]'), base]
    for u in tries:
        try:
            d = json.loads(http_get(u).read())
            hits = d.get('hits', [])
            if hits:
                break
        except Exception:
            hits = []
    out = []
    for h in hits:
        out.append({'slug': h.get('slug'), 'title': h.get('title'),
                    'desc': (h.get('description') or '')[:150],
                    'downloads': h.get('downloads'), 'icon': h.get('icon_url')})
    return out


# ---------------- backups ----------------
def backups_list():
    out = []
    try:
        if os.path.isdir(DRIVE_BACKUPS):
            for n in os.listdir(DRIVE_BACKUPS):
                if n.endswith('.zip'):
                    p = os.path.join(DRIVE_BACKUPS, n)
                    st = os.stat(p)
                    out.append({'name': n, 'size': st.st_size,
                                'mtime': datetime.fromtimestamp(st.st_mtime).strftime('%d %b %H:%M')})
    except Exception:
        pass
    out.sort(key=lambda x: x['mtime'], reverse=True)
    return out


# ---------------- PAGE ----------------
PAGE = """<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>MC Panel v5</title>
<style>
:root{--bg:#0b1220;--card:#121b2e;--card2:#0e1626;--bd:#22304a;--tx:#e2e8f0;--mut:#8fa3bf;--ac:#22d3ee;--gr:#22c55e;--rd:#ef4444;--yl:#f59e0b;--bl:#3b82f6}
*{box-sizing:border-box;-webkit-tap-highlight-color:transparent}
body{font-family:system-ui,-apple-system,sans-serif;background:var(--bg);color:var(--tx);margin:0;padding:0;padding-bottom:40px}
header{position:sticky;top:0;z-index:20;background:linear-gradient(180deg,#0e1728,#0b1220);border-bottom:1px solid var(--bd);padding:10px 12px;display:flex;align-items:center;gap:8px;flex-wrap:wrap}
.brand{font-weight:800;font-size:16px;margin-right:auto}
.ver{font-size:11px;color:var(--mut);font-weight:600}
.pill{font-size:12px;font-weight:700;padding:5px 10px;border-radius:999px;background:#1c2a44;color:var(--mut)}
.pill.on{background:#0c2c1a;color:#4ade80}.pill.off{background:#33121a;color:#f87171}
nav{display:flex;gap:6px;overflow-x:auto;padding:8px 10px;position:sticky;top:52px;z-index:19;background:var(--bg);border-bottom:1px solid var(--bd);scrollbar-width:none}
nav::-webkit-scrollbar{display:none}
.tabbtn{flex:0 0 auto;background:#16223a;color:var(--mut);border:1px solid var(--bd);border-radius:10px;padding:7px 12px;font-size:13px;font-weight:600}
.tabbtn.on{background:#0e2f3a;color:#67e8f9;border-color:#155e75}
main{max-width:640px;margin:0 auto;padding:12px}
.tab{display:block}
.card{background:var(--card);border:1px solid var(--bd);border-radius:14px;padding:14px;margin:10px 0}
.card h3{margin:0 0 10px;font-size:14px;color:#bcd0ea}
.row{display:flex;gap:8px;flex-wrap:wrap;align-items:center}
.grid2{display:grid;grid-template-columns:1fr 1fr;gap:8px}
.kv{background:var(--card2);border:1px solid var(--bd);border-radius:10px;padding:8px 10px}
.kv .k{font-size:11px;color:var(--mut)}
.kv .v{font-size:14px;font-weight:700;margin-top:2px;word-break:break-all}
button{border:0;border-radius:10px;padding:10px 13px;font-size:14px;font-weight:700;cursor:pointer}
.b-gr{background:#16a34a;color:#fff}.b-rd{background:#dc2626;color:#fff}.b-yl{background:#d97706;color:#fff}
.b-bl{background:#2563eb;color:#fff}.b-ac{background:#0891b2;color:#fff}.b-gy{background:#2a3a58;color:#dbe7f7}
button:disabled{opacity:.45}
input,textarea,select{width:100%;padding:10px;border-radius:10px;border:1px solid var(--bd);background:#0a1322;color:var(--tx);font-size:14px}
textarea{font-family:ui-monospace,monospace;font-size:12.5px;min-height:240px}
pre{background:#060b14;border:1px solid var(--bd);border-radius:10px;padding:10px;font-size:12px;overflow:auto;max-height:320px;white-space:pre-wrap;margin:0;font-family:ui-monospace,monospace}
.cl{color:#9fb4d0}.cl.err{color:#f87171}.cl.warn{color:#fbbf24}.cl.chat{color:#7dd3fc}
.chatbox{max-height:340px;overflow:auto;display:flex;flex-direction:column;gap:6px}
.bub{max-width:85%;padding:7px 10px;border-radius:12px;font-size:13.5px;line-height:1.35}
.bub .u{font-size:11px;font-weight:700;margin-bottom:2px}
.bub.chat{align-self:flex-start;background:#16233c;border:1px solid #22304a}
.bub.mine{align-self:flex-end;background:#0d3b46;border:1px solid #155e75}
.bub.sys{align-self:center;background:transparent;color:var(--mut);font-size:12px;border:1px dashed #22304a}
.mut{color:var(--mut);font-size:12px;line-height:1.5}
.frow{display:flex;align-items:center;gap:8px;padding:9px 8px;border-bottom:1px solid #1a2740}
.frow:last-child{border-bottom:0}
.fname{flex:1;min-width:0;overflow:hidden;text-overflow:ellipsis;white-space:nowrap;font-size:13.5px}
.fmeta{font-size:11px;color:var(--mut);margin-top:2px}
.ibtn{background:#1b2a45;color:#cfe0f5;border-radius:9px;padding:7px 9px;font-size:13px;border:1px solid var(--bd)}
.chip{display:inline-block;background:#16324a;color:#7dd3fc;border-radius:999px;padding:5px 10px;font-size:12.5px;margin:3px 4px 0 0}
.chipsrow{display:flex;flex-wrap:wrap}
canvas{width:100%;height:74px;display:block}
.modal{position:fixed;inset:0;background:#000b;z-index:50;display:flex;align-items:center;justify-content:center;padding:14px}
.modal .inner{background:var(--card);border:1px solid var(--bd);border-radius:14px;padding:14px;width:100%;max-width:620px;max-height:92vh;overflow:auto}
a{color:#67e8f9;word-break:break-all}
.mrres{display:flex;gap:10px;align-items:center;padding:9px 0;border-bottom:1px solid #1a2740}
.mrres img{width:34px;height:34px;border-radius:8px;background:#222}
.mrres .t{font-weight:700;font-size:13.5px}
.hidden{display:none}
</style>
</head>
<body>
<header>
  <div class="brand">MC Panel <span class="ver">v5</span></div>
  <span class="pill" id="pill">check...</span>
  <span class="pill" id="pillPl">0/0 players</span>
</header>
<nav>
  <button class="tabbtn on" data-tab="dash">Dashboard</button>
  <button class="tabbtn" data-tab="console">Console</button>
  <button class="tabbtn" data-tab="chat">Chat</button>
  <button class="tabbtn" data-tab="files">Files</button>
  <button class="tabbtn" data-tab="plugins">Plugins</button>
  <button class="tabbtn" data-tab="backups">Backups</button>
  <button class="tabbtn" data-tab="software">Software</button>
</nav>
<main>
  <section class="tab" id="tab-dash">
    <div class="card">
      <div class="row">
        <button class="b-gr" onclick="ctl('start')">Start</button>
        <button class="b-rd" onclick="ctl('stop')">Stop</button>
        <button class="b-yl" onclick="ctl('restart')">Restart</button>
        <span class="mut" id="up" style="margin-left:auto"></span>
      </div>
    </div>
    <div class="card">
      <div class="grid2">
        <div class="kv"><div class="k">Players</div><div class="v" id="stPlayers">-</div></div>
        <div class="kv"><div class="k">TPS</div><div class="v" id="stTps">-</div></div>
        <div class="kv"><div class="k">CPU</div><div class="v" id="stCpu">-</div></div>
        <div class="kv"><div class="k">Minecraft RAM</div><div class="v" id="stRam">-</div></div>
        <div class="kv"><div class="k">System RAM</div><div class="v" id="stSys">-</div></div>
        <div class="kv"><div class="k">MSPT</div><div class="v" id="stMspt">-</div></div>
        <div class="kv"><div class="k">Version</div><div class="v" id="stVer">-</div></div>
        <div class="kv"><div class="k">Software</div><div class="v" id="stSoft">-</div></div>
      </div>
    </div>
    <div class="card">
      <h3>Performance (live)</h3>
      <div class="mut">CPU %</div><canvas id="chCpu"></canvas>
      <div class="mut" style="margin-top:6px">Minecraft RAM (MB)</div><canvas id="chRam"></canvas>
      <div class="mut" style="margin-top:6px">TPS (20 = perfect)</div><canvas id="chTps"></canvas>
    </div>
    <div class="card">
      <h3>Online Players</h3>
      <div class="chipsrow" id="playersBox"><span class="mut">-</span></div>
    </div>
    <div class="card">
      <h3>Panel Link</h3>
      <div class="mut" id="panelLinkBox">-</div>
      <div class="mut" style="margin-top:6px">Backup: <span id="stBackup">-</span> | Auto-save: har 60 sec + turant stop/restart par</div>
      <div class="mut" id="sysSt" style="margin-top:4px"></div>
    </div>
    <div class="card">
      <h3>Live Activity</h3>
      <div id="eventsBox" class="mut">-</div>
    </div>
    <div class="card">
      <h3>Quick Commands</h3>
      <div class="row">
        <button class="ibtn" onclick="q('time set day')">Time: Day</button>
        <button class="ibtn" onclick="q('weather clear')">Weather Clear</button>
        <button class="ibtn" onclick="q('save-all')">Save</button>
        <button class="ibtn" onclick="q('list')">List</button>
        <button class="ibtn" onclick="q('tps')">TPS</button>
      </div>
      <div class="row" style="margin-top:8px">
        <input id="dashCmd" placeholder="koi bhi command (bina /)">
        <button class="b-bl" onclick="q(document.getElementById('dashCmd').value)">Send</button>
      </div>
    </div>
  </section>

  <section class="tab hidden" id="tab-console">
    <div class="card">
      <div class="row">
        <label class="mut"><input type="checkbox" id="logClean" style="width:auto"> clean view</label>
        <label class="mut"><input type="checkbox" id="logAuto" checked style="width:auto"> auto-scroll</label>
        <a class="mut" style="margin-left:auto" href="/api/file/download?path=server_console.log">download log</a>
      </div>
    </div>
    <div class="card"><pre id="logBox">loading...</pre></div>
    <div class="card">
      <div class="row">
        <input id="cmdInput" placeholder="command likho... (jaise: say hello)">
        <button class="b-bl" onclick="sendCmd()">Send</button>
      </div>
      <div class="row" style="margin-top:8px">
        <button class="ibtn" onclick="cmdLast()">last command</button>
        <button class="ibtn" onclick="q('save-all flush')">save-all</button>
        <button class="ibtn" onclick="q('whitelist list')">whitelist</button>
        <button class="ibtn" onclick="q('op')">op help</button>
      </div>
    </div>
  </section>

  <section class="tab hidden" id="tab-chat">
    <div class="card">
      <div class="row">
        <input id="sayInput" placeholder="server se message bhejo (say)">
        <button class="b-ac" onclick="sendSay()">Send</button>
      </div>
      <div class="mut" style="margin-top:6px">Server chat live - players ke messages, joins, advancements sab yahan.</div>
    </div>
    <div class="card"><div class="chatbox" id="chatBox">loading...</div></div>
  </section>

  <section class="tab hidden" id="tab-files">
    <div class="card">
      <div class="row">
        <button class="ibtn" onclick="loadFiles(fParent)">.. up</button>
        <button class="ibtn" onclick="loadFiles(fPathNow)">refresh</button>
        <span class="mut" id="fPath" style="margin-left:auto"></span>
      </div>
      <div class="row" style="margin-top:8px">
        <button class="ibtn" onclick="mkDir()">+ folder</button>
        <label class="ibtn" style="cursor:pointer">upload<input type="file" id="fUp" class="hidden" onchange="doUpload()"></label>
      </div>
    </div>
    <div class="card" id="fList">loading...</div>
  </section>

  <section class="tab hidden" id="tab-plugins">
    <div class="card">
      <h3>Installed Plugins</h3>
      <div id="plList" class="mut">loading...</div>
      <div class="mut" style="margin-top:8px">Note: plugin on/off karne ke baad server Restart karo (Dashboard me).</div>
    </div>
    <div class="card">
      <h3>Install from Modrinth</h3>
      <div class="chipsrow">
        <span class="chip" onclick="mrChip('luckperms')">LuckPerms</span>
        <span class="chip" onclick="mrChip('viaversion')">ViaVersion</span>
        <span class="chip" onclick="mrChip('worldedit')">WorldEdit</span>
        <span class="chip" onclick="mrChip('chunky')">Chunky</span>
        <span class="chip" onclick="mrChip('spark')">Spark</span>
        <span class="chip" onclick="mrChip('coreprotect')">CoreProtect</span>
        <span class="chip" onclick="mrChip('geyser')">Geyser</span>
        <span class="chip" onclick="mrChip('simple-voice-chat')">Voice Chat</span>
      </div>
      <div class="row" style="margin-top:8px">
        <input id="mrQ" placeholder="plugin dhundo (Modrinth)">
        <button class="b-ac" onclick="mrGo()">Search</button>
      </div>
      <div id="mrRes"></div>
    </div>
    <div class="card">
      <h3>Jar URL se install</h3>
      <div class="row">
        <input id="urlJar" placeholder="https://.../plugin.jar">
        <button class="b-bl" onclick="urlInstall()">Install</button>
      </div>
      <div class="row" style="margin-top:8px">
        <label class="ibtn" style="cursor:pointer">ya jar upload karo<input type="file" id="upJar" class="hidden" onchange="jarUpload()"></label>
        <span class="mut">file phone se seedha plugins/ me jayegi</span>
      </div>
    </div>
  </section>

  <section class="tab hidden" id="tab-backups">
    <div class="card">
      <h3>Backups</h3>
      <div class="mut" id="bkSt">loading...</div>
      <div class="row" style="margin-top:8px">
        <button class="b-gr" onclick="bkNow()">Backup Now</button>
      </div>
    </div>
    <div class="card" id="bkList">loading...</div>
  </section>

  <section class="tab hidden" id="tab-software">
    <div class="card">
      <h3>Server Software</h3>
      <div class="mut">Current: <b id="swCur">-</b> | MC version: <span id="swVer">-</span></div>
      <div class="mut" id="swSt" style="margin-top:8px"></div>
    </div>
    <div class="card" id="swOpts">loading...</div>
    <div class="mut">Software badalne par server restart hoga. Paper/Purpur = plugins chalte hain. Vanilla/Fabric = plugins band ho jayenge (Fabric me mods chalte hain).</div>
  </section>
</main>
<div class="modal hidden" id="modal">
  <div class="inner">
    <div class="row"><b id="edTitle" style="margin-right:auto">edit</b>
      <button class="b-gy" onclick="closeEdit()">close</button>
      <button class="b-gr" onclick="saveEdit()">Save</button>
    </div>
    <textarea id="edArea" spellcheck="false"></textarea>
  </div>
</div>
<script>
function api(p,o){return fetch(p,o||{}).then(function(r){if(!r.ok)throw new Error('HTTP '+r.status);return r.json()});}
function post(p,b){return api(p,{method:'POST',headers:{'Content-Type':'application/json'},body:JSON.stringify(b||{})});}
function esc(s){return String(s==null?'':s).replace(/[&<>"']/g,function(c){return {'&':'&amp;','<':'&lt;','>':'&gt;','"':'&quot;',"'":'&#39;'}[c];});}
function ja(s){return esc(JSON.stringify(String(s)));}
function sz(n){return n<1024?n+' B':(n<1048576?(n/1024).toFixed(1)+' KB':(n/1048576).toFixed(1)+' MB');}
var TAB='dash';
document.querySelectorAll('.tabbtn').forEach(function(b){b.onclick=function(){showTab(b.getAttribute('data-tab'));};});
function showTab(t){TAB=t;
 document.querySelectorAll('.tabbtn').forEach(function(b){b.className='tabbtn'+(b.getAttribute('data-tab')===t?' on':'');});
 document.querySelectorAll('.tab').forEach(function(s){s.className='tab'+((s.id==='tab-'+t)?'':' hidden');});
 if(t==='files')loadFiles(fPathNow);
 if(t==='plugins')loadPlugins();
 if(t==='backups')loadBackups();
 if(t==='software')loadSoftware();
 if(t==='chat')pollChat();
}
function ctl(c){if(c==='stop'&&!confirm('Server band karna hai?'))return;post('/api/control',{command:c}).then(pollStatus);}
function q(c){if(!c)return;post('/api/command',{command:c}).then(pollStatus);document.getElementById('dashCmd').value='';}
function sendCmd(){var i=document.getElementById('cmdInput');if(!i.value)return;q(i.value);i.value='';}
function cmdLast(){document.getElementById('cmdInput').value=localStorage.getItem('mclast')||'';}
document.getElementById('cmdInput').addEventListener('keydown',function(e){if(e.key==='Enter'){localStorage.setItem('mclast',this.value);sendCmd();}});
function sendSay(){var i=document.getElementById('sayInput');if(!i.value)return;q('say '+i.value);i.value='';}
var stTimer=null,lsTps=0;
function pollStatus(){api('/api/status').then(function(s){
 var pill=document.getElementById('pill');
 pill.textContent=s.running?('ONLINE '+(s.software||'')):'OFFLINE';
 pill.className='pill '+(s.running?'on':'off');
 document.getElementById('pillPl').textContent=s.players+'/'+(s.maxplayers||'?')+' players';
 document.getElementById('stPlayers').textContent=s.players+' online';
 document.getElementById('stTps').textContent=(s.tps==null?'-':s.tps);
 document.getElementById('stCpu').textContent=s.cpu+' %';
 document.getElementById('stRam').textContent=s.ram_mb+' MB';
 document.getElementById('stSys').textContent=s.sys_ram+' %';
 document.getElementById('stMspt').textContent=(s.mspt==null?'-':s.mspt+' ms');
 document.getElementById('stVer').textContent=s.mc_version;
 document.getElementById('stSoft').textContent=s.software;
 document.getElementById('stBackup').textContent=s.backup;
 document.getElementById('up').textContent='uptime: '+(s.uptime||'-');
 var pb=document.getElementById('playersBox');
 pb.innerHTML=(s.players_list&&s.players_list.length)?s.players_list.map(function(n){return '<span class="chip">'+esc(n)+'</span>';}).join(''):'<span class="mut">koi nahi online</span>';
 document.getElementById('panelLinkBox').innerHTML=s.url_cf?('<a href="'+esc(s.url_cf)+'/?key='+encodeURIComponent(s.panel_key||'')+'">'+esc(s.url_cf)+'</a>'):'starting...';
 var sy=[];if(s.restore_status)sy.push('Restore: '+s.restore_status);if(s.backup_status)sy.push('Backup: '+s.backup_status);
 document.getElementById('sysSt').textContent=sy.join(' | ');
}).catch(function(e){document.getElementById('pill').className='pill off';document.getElementById('pill').textContent='panel error';});}
function draw(cv,arr,color,maxv){if(!cv)return;var dpr=window.devicePixelRatio||1,w=cv.clientWidth||300,h=74;cv.width=w*dpr;cv.height=h*dpr;var c=cv.getContext('2d');c.scale(dpr,dpr);c.clearRect(0,0,w,h);
 var n=arr.length;if(n<2)return;var m=maxv||0;for(var i=0;i<n;i++){if(arr[i]>m)m=arr[i];}m=m||1;m=m*1.08;
 c.beginPath();c.moveTo(0,h-2);for(var i=0;i<n;i++){var x=i*(w-2)/(n-1)+1;var y=h-4-((arr[i]/m)*(h-10));c.lineTo(x,y);}c.lineTo(w-1,h-2);c.closePath();
 c.fillStyle=color+'22';c.fill();c.beginPath();for(var i=0;i<n;i++){var x=i*(w-2)/(n-1)+1;var y=h-4-((arr[i]/m)*(h-10));if(i===0)c.moveTo(x,y);else c.lineTo(x,y);}
 c.strokeStyle=color;c.lineWidth=2;c.stroke();c.fillStyle=color;c.font='11px system-ui';c.fillText(String(arr[n-1]),4,12);}
function pollStats(){api('/api/stats').then(function(s){
 draw(document.getElementById('chCpu'),s.history.cpu,'#22d3ee',100);
 draw(document.getElementById('chRam'),s.history.ram,'#a78bfa');
 draw(document.getElementById('chTps'),s.history.tps,'#4ade80',20);
}).catch(function(){});}
var lastLogHtml='',lastChatN=0,lastEvtN=0;
function pollConsole(){if(TAB!=='console')return;api('/api/log').then(function(d){
 var lines=d.log.split('\\n');var clean=document.getElementById('logClean').checked;
 if(clean){lines=lines.filter(function(l){return !/TPS from last|There are \\d+ of a max|Saved the game|Saving the game|save-all|Running AutoSave|autosave/i.test(l);});}
 lines=lines.slice(-400);
 var html=lines.map(function(l){var cl='cl';if(/ERROR|SEVERE|Exception|FAIL/i.test(l))cl+=' err';else if(/WARN|Could not/i.test(l))cl+=' warn';else if(/<[^>]+> /.test(l))cl+=' chat';return '<span class="'+cl+'">'+esc(l)+'</span>';}).join('\\n');
 if(html!==lastLogHtml){lastLogHtml=html;var b=document.getElementById('logBox');b.innerHTML=html;if(document.getElementById('logAuto').checked)b.scrollTop=b.scrollHeight;}
}).catch(function(){});}
function pollChat(){api('/api/chat').then(function(d){
 if(d.items.length===lastChatN)return;lastChatN=d.items.length;
 var box=document.getElementById('chatBox');box.innerHTML=d.items.map(function(m){
  if(m.kind==='chat'){return '<div class="bub chat"><div class="u">'+esc(m.user)+' <span class="mut">'+esc(m.time)+'</span></div>'+esc(m.text)+'</div>';}
  if(m.kind==='server'){return '<div class="bub mine"><div class="u">Server <span class="mut">'+esc(m.time)+'</span></div>'+esc(m.text)+'</div>';}
  return '<div class="bub sys">'+esc(m.user+' '+m.text+' ('+m.time+')')+'</div>';}).join('')||'<span class="mut">abhi koi chat nahi</span>';
 box.scrollTop=box.scrollHeight;
 var ev=document.getElementById('eventsBox');
 var evs=d.items.filter(function(m){return m.kind!=='chat';}).slice(-6).reverse();
 lastEvtN=d.items.length;
 ev.innerHTML=evs.map(function(m){return '<div>'+esc(m.time+' - '+m.user+' '+m.text)+'</div>';}).join('')||'<span class="mut">-</span>';
}).catch(function(){});}
var fPathNow='',fParent='';
function fIcon(e){if(e.dir)return '\\uD83D\\uDCC1';var n=e.name.toLowerCase();if(n.endsWith('.jar'))return '\\uD83E\\uDDE9';if(n.endsWith('.log')||n.endsWith('.txt'))return '\\uD83D\\uDCC4';if(n.endsWith('.json')||n.endsWith('.yml')||n.endsWith('.yaml')||n.endsWith('.properties'))return '\\u2699\\uFE0F';if(n.endsWith('.zip'))return '\\uD83D\\uDCE6';return '\\uD83D\\uDCC4';}
function loadFiles(p){api('/api/files?path='+encodeURIComponent(p||'')).then(function(d){
 fPathNow=d.path;fParent=d.parent||'';
 document.getElementById('fPath').textContent='/'+d.path+' ('+d.entries.length+')';
 document.getElementById('fList').innerHTML=d.entries.map(function(e){
  var full=(fPathNow?fPathNow+'/':'')+e.name;
  var head=e.dir?('<button class="ibtn" onclick="loadFiles('+ja(full)+')">'+fIcon(e)+' '+esc(e.name)+'</button>')
                :(fIcon(e)+' '+esc(e.name));
  var acts=e.dir?'':('<button class="ibtn" onclick="editFile('+ja(full)+')">edit</button>'+
                     '<a class="ibtn" href="/api/file/download?path='+encodeURIComponent(full)+'">get</a>');
  return '<div class="frow"><div class="fname">'+head+'<div class="fmeta">'+esc(e.mtime)+(e.dir?'':' - '+sz(e.size))+'</div></div>'+acts+
   '<button class="ibtn" onclick="delFile('+ja(full)+','+(e.dir?'true':'false')+')">del</button></div>';
 }).join('')||'<span class="mut">khali folder</span>';
}).catch(function(e){document.getElementById('fList').innerHTML='<span class="mut">error: '+esc(e.message)+'</span>';});}
var edPath='';
function editFile(p){api('/api/file/read?path='+encodeURIComponent(p)).then(function(d){
 if(d.binary){alert('binary file - edit nahi ho sakti, download karo');return;}
 edPath=p;document.getElementById('edTitle').textContent='/'+p;document.getElementById('edArea').value=d.content;document.getElementById('modal').className='modal';});
}
function closeEdit(){document.getElementById('modal').className='modal hidden';}
function saveEdit(){post('/api/file/write',{path:edPath,content:document.getElementById('edArea').value}).then(function(){closeEdit();loadFiles(fPathNow);alert('save ho gaya');});}
function delFile(p,isDir){if(!confirm('DELETE: '+p+' ?'))return;post('/api/file/delete',{path:p,recursive:isDir}).then(function(){loadFiles(fPathNow);});}
function mkDir(){var n=prompt('naya folder ka naam:');if(!n)return;post('/api/file/mkdir',{path:(fPathNow?fPathNow+'/':'')+n}).then(function(){loadFiles(fPathNow);});}
function doUpload(){var f=document.getElementById('fUp').files[0];if(!f)return;var fd=new FormData();fd.append('path',fPathNow);fd.append('file',f);fetch('/api/upload',{method:'POST',body:fd}).then(function(r){return r.json();}).then(function(){document.getElementById('fUp').value='';loadFiles(fPathNow);});}
var hyp='';
function loadPlugins(){api('/api/plugins').then(function(d){
 document.getElementById('plList').innerHTML=d.items.map(function(p){return '<div class="frow"><div class="fname">'+esc(p.name)+'<div class="fmeta">'+sz(p.size)+' - '+(p.enabled?'ON':'OFF')+'</div></div>'+
 '<button class="ibtn" onclick="plToggle('+ja(p.name)+')">'+(p.enabled?'disable':'enable')+'</button>'+
 '<button class="ibtn" onclick="plDel('+ja(p.name)+')">del</button></div>';}).join('')||'<span class="mut">koi plugin nahi - niche se install karo</span>';
});}
function plToggle(n){post('/api/plugin/toggle',{name:n}).then(loadPlugins);}
function plDel(n){if(!confirm('plugin delete: '+n+' ?'))return;post('/api/plugin/delete',{name:n}).then(loadPlugins);}
function mrChip(s){document.getElementById('mrQ').value=s;mrGo();}
function mrGo(){var qq=document.getElementById('mrQ').value;if(!qq)return;document.getElementById('mrRes').innerHTML='<div class="mut">search...</div>';
 api('/api/modrinth/search?q='+encodeURIComponent(qq)).then(function(d){
 document.getElementById('mrRes').innerHTML=d.results.map(function(p){return '<div class="mrres">'+(p.icon?'<img src="'+esc(p.icon)+'">':'')+'<div style="flex:1;min-width:0"><div class="t">'+esc(p.title)+'</div><div class="mut">'+esc(p.desc)+'</div></div><button class="b-ac" onclick="mrInstall('+ja(p.slug)+',this)">Install</button></div>';}).join('')||'<div class="mut">kuch nahi mila</div>';});}
function mrInstall(slug,btn){btn.disabled=true;btn.textContent='...';post('/api/plugin/install',{project:slug}).then(function(d){btn.textContent=d.ok?'INSTALLED':'FAIL';loadPlugins();}).catch(function(e){btn.textContent='FAIL';});}
function urlInstall(){var u=document.getElementById('urlJar').value;if(!u)return;post('/api/plugin/install-url',{url:u}).then(function(){document.getElementById('urlJar').value='';loadPlugins();}).catch(function(e){alert('fail: '+e.message);});}
function jarUpload(){var f=document.getElementById('upJar').files[0];if(!f)return;var fd=new FormData();fd.append('path','plugins');fd.append('file',f);fetch('/api/upload',{method:'POST',body:fd}).then(function(r){return r.json();}).then(function(){document.getElementById('upJar').value='';loadPlugins();});}
function loadBackups(){api('/api/backups').then(function(d){
 document.getElementById('bkSt').innerHTML='Drive: '+(d.drive?'ON':'OFF')+' | last auto-sync: <b>'+esc(d.last_sync||'-')+'</b> | every 60s'+(d.backup_status?' | '+esc(d.backup_status):'')+(d.restore_status?' | '+esc(d.restore_status):'');
 document.getElementById('bkList').innerHTML=d.items.map(function(b){return '<div class="frow"><div class="fname">'+esc(b.name)+'<div class="fmeta">'+esc(b.mtime)+' - '+sz(b.size)+'</div></div>'+
 '<a class="ibtn" href="/api/backup/download?name='+encodeURIComponent(b.name)+'">get</a>'+
 '<button class="ibtn" onclick="bkRestore('+ja(b.name)+')">restore</button></div>';}).join('')||'<span class="mut">koi backup nahi - Backup Now dabao</span>';
});}
function bkNow(){post('/api/backup/create').then(function(){loadBackups();alert('backup banao request bhej di - 30-60 sec me list me dikhega');});}
function bkRestore(n){if(!confirm('RESTORE '+n+' ?\nServer band hoga, world is backup se replace hogi.'))return;if(!confirm('PAKKA? Current world ka safety backup pehle banega, phir restore.'))return;
 post('/api/backup/restore',{name:n}).then(function(){loadBackups();alert('restore start - 1-2 min lagenge, panel me status dekho');});}
function loadSoftware(){api('/api/software').then(function(d){
 document.getElementById('swCur').textContent=d.current;document.getElementById('swVer').textContent=d.mc_version;
 document.getElementById('swSt').textContent=d.status||'';
 document.getElementById('swOpts').innerHTML=d.options.map(function(o){return '<div class="frow"><div class="fname"><b>'+esc(o.label)+'</b>'+(o.current?' (current)':'')+'<div class="fmeta">'+esc(o.desc)+'</div></div>'+(o.current?'':'<button class="b-yl" onclick="swSet('+ja(o.id)+')">Apply</button>')+'</div>';}).join('');
});}
function swSet(id){if(!confirm('Software badal kar '+id+' karun?\nServer restart hoga aur jar download hoga.'))return;post('/api/software',{id:id}).then(function(){loadSoftware();setTimeout(loadSoftware,3000);});}
setInterval(pollStatus,5000);setInterval(pollStats,5000);setInterval(pollConsole,2500);setInterval(function(){if(TAB==='chat')pollChat();},3000);setInterval(function(){if(TAB==='backups')loadBackups();},15000);setInterval(function(){if(TAB==='software')loadSoftware();},5000);
pollStatus();pollStats();pollConsole();pollChat();
</script>
</body>
</html>"""


# ---------------- routes ----------------
@app.route('/')
def index():
    return PAGE


@app.route('/api/status')
def api_status():
    up = '-'
    try:
        if is_running():
            ts = float(read_file('SERVER_STARTED_AT', '0') or 0)
            if ts > 0:
                mins = int((time.time() - ts) // 60)
                up = str(mins // 60) + 'h ' + str(mins % 60) + 'm'
    except Exception:
        pass
    return jsonify({
        'running': is_running(),
        'playit': _u('playit', exact=True),
        'cloudflared': _u('cloudflared'),
        'url_cf': read_file('CF_URL'),
        'panel_key': KEY if request.cookies.get('mcpanel') or _key_ok(request.args.get('key', '')) else '',
        'backup': read_file('LAST_BACKUP', '-'),
        'players': PLAYERS['count'],
        'maxplayers': PLAYERS['max'],
        'players_list': PLAYERS['names'],
        'tps': LATEST['tps'],
        'mspt': LATEST['mspt'],
        'cpu': LATEST['cpu'],
        'ram_mb': LATEST['ram_mb'],
        'sys_ram': LATEST['sys_ram'],
        'software': read_file('SOFTWARE', 'paper') or 'paper',
        'mc_version': read_file('MC_VERSION', '-'),
        'uptime': up,
        'backup_status': read_file('BACKUP_STATUS'),
        'restore_status': read_file('RESTORE_STATUS'),
        'drive': os.path.isdir(DRIVE_MC),
        'time': datetime.now().strftime('%H:%M:%S'),
    })


@app.route('/api/stats')
def api_stats():
    return jsonify({'history': HIST, 'latest': LATEST})


@app.route('/api/log')
def api_log():
    return jsonify({'log': tail_bytes('server_console.log', 60000)})


@app.route('/api/chat')
def api_chat():
    return jsonify({'items': parse_chat()})


@app.route('/api/command', methods=['POST'])
def api_command():
    c = (request.json or {}).get('command', '').replace('\n', ' ').strip()
    if c:
        stuff(c)
    return jsonify({'ok': True})


@app.route('/api/control', methods=['POST'])
def api_control():
    c = (request.json or {}).get('command', '')
    if c == 'start':
        threading.Thread(target=do_start, daemon=True).start()
    elif c == 'stop':
        threading.Thread(target=do_stop, daemon=True).start()
    elif c == 'restart':
        threading.Thread(target=do_restart, daemon=True).start()
    return jsonify({'ok': True})


# ----- files -----
@app.route('/api/files')
def api_files():
    try:
        return jsonify(list_dir(request.args.get('path', '')))
    except Exception as e:
        return jsonify({'error': str(e)}), 400


@app.route('/api/file/read')
def api_file_read():
    try:
        p = safe_path(request.args.get('path', ''))
        if not os.path.isfile(p):
            return jsonify({'error': 'file nahi mili'}), 404
        if os.path.getsize(p) > 3 * 1024 * 1024:
            return jsonify({'error': 'file bahut badi (3MB+) - download karo'}), 400
        raw = open(p, 'rb').read()
        try:
            return jsonify({'content': raw.decode('utf-8'), 'binary': False})
        except Exception:
            return jsonify({'content': '', 'binary': True})
    except Exception as e:
        return jsonify({'error': str(e)}), 400


@app.route('/api/file/write', methods=['POST'])
def api_file_write():
    d = request.json or {}
    try:
        p = safe_path(d.get('path', ''))
        if os.path.isdir(p):
            return jsonify({'error': 'ye folder hai'}), 400
        os.makedirs(os.path.dirname(p), exist_ok=True)
        tmp = p + '.tmp'
        with open(tmp, 'w', encoding='utf-8') as f:
            f.write(d.get('content', ''))
        os.replace(tmp, p)
        return jsonify({'ok': True})
    except Exception as e:
        return jsonify({'error': str(e)}), 400


@app.route('/api/file/delete', methods=['POST'])
def api_file_delete():
    d = request.json or {}
    try:
        p = safe_path(d.get('path', ''))
        if p == BASE:
            return jsonify({'error': 'base delete nahi hota'}), 400
        if os.path.isdir(p):
            if d.get('recursive'):
                shutil.rmtree(p)
            else:
                os.rmdir(p)
        else:
            os.remove(p)
        return jsonify({'ok': True})
    except Exception as e:
        return jsonify({'error': str(e)}), 400


@app.route('/api/file/mkdir', methods=['POST'])
def api_file_mkdir():
    d = request.json or {}
    try:
        os.makedirs(safe_path(d.get('path', '')), exist_ok=True)
        return jsonify({'ok': True})
    except Exception as e:
        return jsonify({'error': str(e)}), 400


@app.route('/api/file/download')
def api_file_download():
    try:
        p = safe_path(request.args.get('path', ''))
        if os.path.isfile(p):
            return send_file(p, as_attachment=True, download_name=os.path.basename(p))
        if os.path.isdir(p):
            os.makedirs(TMP, exist_ok=True)
            name = (os.path.basename(p) or 'folder') + '-' + datetime.now().strftime('%H%M%S')
            zp = shutil.make_archive(os.path.join(TMP, name), 'zip', root_dir=p)
            return send_file(zp, as_attachment=True, download_name=name + '.zip')
        return jsonify({'error': 'nahi mila'}), 404
    except Exception as e:
        return jsonify({'error': str(e)}), 400


@app.route('/api/upload', methods=['POST'])
def api_upload():
    try:
        rel = request.form.get('path', '')
        target = safe_path(rel)
        os.makedirs(target, exist_ok=True)
        f = request.files.get('file')
        if not f or not f.filename:
            return jsonify({'error': 'file nahi mili'}), 400
        name = _plug_name(f.filename)
        dest = os.path.join(target, name)
        tmp = dest + '.new'
        f.save(tmp)
        os.replace(tmp, dest)
        return jsonify({'ok': True, 'name': name, 'size': os.path.getsize(dest)})
    except Exception as e:
        return jsonify({'error': str(e)}), 400


# ----- plugins -----
@app.route('/api/plugins')
def api_plugins():
    return jsonify({'items': plugin_list()})


@app.route('/api/plugin/toggle', methods=['POST'])
def api_plugin_toggle():
    d = request.json or {}
    try:
        n = _plug_name(d.get('name', ''))
        p = os.path.join(plugins_dir(), n)
        if not os.path.exists(p):
            return jsonify({'error': 'plugin nahi mila'}), 404
        if n.endswith('.jar.disabled'):
            os.rename(p, p[:-9])
        elif n.endswith('.jar'):
            os.rename(p, p + '.disabled')
        return jsonify({'ok': True})
    except Exception as e:
        return jsonify({'error': str(e)}), 400


@app.route('/api/plugin/delete', methods=['POST'])
def api_plugin_delete():
    d = request.json or {}
    try:
        n = _plug_name(d.get('name', ''))
        p = os.path.join(plugins_dir(), n)
        if os.path.exists(p):
            os.remove(p)
        return jsonify({'ok': True})
    except Exception as e:
        return jsonify({'error': str(e)}), 400


@app.route('/api/plugin/install-url', methods=['POST'])
def api_plugin_install_url():
    d = request.json or {}
    try:
        url = d.get('url', '')
        if not url.startswith('http'):
            return jsonify({'error': 'URL sahi nahi'}), 400
        name = d.get('filename') or os.path.basename(urllib.parse.urlparse(url).path) or 'plugin.jar'
        if not name.endswith('.jar'):
            name += '.jar'
        dest = os.path.join(plugins_dir(), _plug_name(name))
        got = download_to(url, dest)
        return jsonify({'ok': True, 'name': os.path.basename(dest), 'size': got})
    except Exception as e:
        return jsonify({'error': str(e)[:200]}), 400


@app.route('/api/plugin/install', methods=['POST'])
def api_plugin_install():
    d = request.json or {}
    try:
        name, got = modrinth_install(d.get('project', ''))
        return jsonify({'ok': True, 'name': name, 'size': got})
    except Exception as e:
        return jsonify({'error': str(e)[:200]}), 400


@app.route('/api/modrinth/search')
def api_modrinth():
    try:
        return jsonify({'results': modrinth_search(request.args.get('q', ''))})
    except Exception as e:
        return jsonify({'error': str(e)[:200], 'results': []}), 400


# ----- backups -----
@app.route('/api/backups')
def api_backups():
    return jsonify({'drive': os.path.isdir(DRIVE_MC), 'items': backups_list(),
                    'last_sync': read_file('LAST_BACKUP', '-'),
                    'backup_status': read_file('BACKUP_STATUS'),
                    'restore_status': read_file('RESTORE_STATUS')})


@app.route('/api/backup/create', methods=['POST'])
def api_backup_create():
    try:
        write_file('BACKUP_REQUEST', str(time.time()))
        return jsonify({'ok': True, 'queued': True})
    except Exception as e:
        return jsonify({'error': str(e)}), 400


@app.route('/api/backup/download')
def api_backup_download():
    try:
        name = os.path.basename(request.args.get('name', ''))
        p = os.path.join(DRIVE_BACKUPS, name)
        if not name.endswith('.zip') or not os.path.isfile(p):
            return jsonify({'error': 'backup nahi mila'}), 404
        return send_file(p, as_attachment=True, download_name=name)
    except Exception as e:
        return jsonify({'error': str(e)}), 400


@app.route('/api/backup/restore', methods=['POST'])
def api_backup_restore():
    d = request.json or {}
    try:
        name = os.path.basename(d.get('name', ''))
        if not name.endswith('.zip') or not os.path.isfile(os.path.join(DRIVE_BACKUPS, name)):
            return jsonify({'error': 'backup nahi mila'}), 404
        open(os.path.join(BASE, FLAG), 'w').write('1')
        write_file('RESTORE_REQUEST', name)
        write_file('RESTORE_STATUS', 'queued: ' + name)
        return jsonify({'ok': True, 'queued': True})
    except Exception as e:
        return jsonify({'error': str(e)}), 400


# ----- software -----
@app.route('/api/software')
def api_software():
    cur = read_file('SOFTWARE', 'paper') or 'paper'
    opts = []
    for pid, info in SOFT_INFO.items():
        opts.append({'id': pid, 'label': info['label'], 'desc': info['desc'], 'current': pid == cur})
    return jsonify({'current': cur, 'options': opts, 'mc_version': read_file('MC_VERSION', '-'),
                    'status': read_file('SOFTWARE_STATUS', '')})


@app.route('/api/software', methods=['POST'])
def api_software_set():
    d = request.json or {}
    pid = d.get('id', '')
    if pid not in SOFT_INFO:
        return jsonify({'error': 'unknown software'}), 400
    if pid == (read_file('SOFTWARE', 'paper') or 'paper') and not d.get('force'):
        return jsonify({'ok': True, 'note': 'already same'})
    software_switch(pid)
    return jsonify({'ok': True, 'queued': True})


def _boot_cleanup():
    try:
        os.makedirs(TMP, exist_ok=True)
        now = time.time()
        for n in os.listdir(TMP):
            p = os.path.join(TMP, n)
            try:
                if now - os.path.getmtime(p) > 3600:
                    os.remove(p)
            except Exception:
                pass
    except Exception:
        pass


_boot_cleanup()
threading.Thread(target=sampler, daemon=True).start()
app.run(host='0.0.0.0', port=8081, threaded=True)
'''
open('panel.py', 'w', encoding='utf-8').write(PANEL_CODE)
print('✅ Panel v5.0 ready (dashboard + console + chat + files + plugins + backups + software)')

# 🎮 playit.gg (Minecraft — FREE, no card! secret upar wale SECRET cell se)
import subprocess, time, os

if not globals().get('PLAYIT_SECRET'):
    print('❌ PLAYIT_SECRET nahi mila — Minecraft tab tak nahi chalega!')
    print('   Fix: upar wala 🔑 SECRET cell dobara Run karo aur box me value paste karo')
    print('   (Value chat me di hai — sirf 1 baar karna hai!)')
    print('   Phir ye cell dobara Run karo. (Panel link phir bhi chalega)')
else:
    if not os.path.exists('playit') or os.path.getsize('playit') < 3 * 1024 * 1024:
        print('⬇️ playit download ho raha hai (~6 MB)...')
        subprocess.run("curl -sL --retry 3 -f https://github.com/playit-cloud/playit-agent/releases/download/v1.0.10/playit-linux-amd64 -o playit && chmod +x playit",
                       shell=True)
    print('✅ playit binary ready')

    subprocess.run("pkill -x playit || true", shell=True)
    time.sleep(1)
    subprocess.Popen(['./playit', '--secret', PLAYIT_SECRET, '--log-path', 'playit.log'],
                     stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    print('🚀 playit agent start (wait 10 sec)...')
    time.sleep(10)

    if os.path.exists('playit.log'):
        print('--- playit.log ---')
        print(open('playit.log').read()[-600:])

    print()
    print('ℹ️ Log me WARN / SessionNotSetup aaye to IGNORE karo — normal hai.')
    print('   Bas "playit connected ... verified" dikhe to agent OK!')
    print()
    print('═' * 52)
    print('📱 playit.gg → Tunnels → Add Tunnel (agar abhi tak nahi banaya):')
    print('   → Type: Minecraft (Java)')
    print('   → Local: 127.0.0.1  Port: 25565')
    print('   → Create → jo address mile WO DOSTO KO DO! 🎮')
    print('   (Address SAME rehta hai — ek baar bana lo, hamesha ke liye)')
    print('═' * 52)

# 🌐 Panel + cloudflared start — panel link yahin se milega!
import subprocess, time, urllib.request, urllib.error, json, os, re

def alive(pattern, exact=False):
    cmd = ['pgrep', '-x', pattern] if exact else ['pgrep', '-f', pattern]
    try:
        return subprocess.run(cmd, capture_output=True).returncode == 0
    except Exception:
        return False

def notify_publish(url):
    try:
        topic = globals().get('NTFY_TOPIC') or ''
        key = globals().get('PANEL_KEY') or ''
        if not topic:
            return False
        full = url + ('/?key=' + key if key else '')
        req = urllib.request.Request('https://ntfy.sh/' + topic,
                                     data=('Panel ready: ' + full).encode(),
                                     headers={'Title': 'MC Panel link 🔗', 'Tags': 'video_game',
                                              'Click': full, 'Priority': 'default'})
        urllib.request.urlopen(req, timeout=10)
        return True
    except Exception:
        return False

# Panel
if not alive('panel.py'):
    subprocess.Popen(['python3', 'panel.py'], stdout=open('panel.log', 'w'), stderr=subprocess.STDOUT)
    print('🌐 Panel start (port 8081)')
time.sleep(3)
try:
    urllib.request.urlopen('http://127.0.0.1:8081/favicon.ico', timeout=5)
    print('✅ Panel v5 live hai (local)')
except urllib.error.HTTPError:
    print('✅ Panel v5 live hai (local)')
except Exception as e:
    print('⚠️ Panel local check:', e)

# cloudflared — PANEL LINK (koi account nahi, koi takrav nahi!)
if not alive('cloudflared'):
    subprocess.Popen(['./cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8081', '--no-autoupdate', '--protocol', 'http2'],
                     stdout=open('cloudflared.log', 'w'), stderr=subprocess.STDOUT)
    print('🌐 cloudflared start (panel link)')

print('⏳ Link ready hone ka wait (~30 sec)...')
CF = ''
for i in range(15):
    time.sleep(3)
    if not CF and os.path.exists('cloudflared.log'):
        try:
            txt = open('cloudflared.log', errors='ignore').read()
            m = [x for x in re.findall(r'https://[a-z0-9-]+\.trycloudflare\.com', txt) if not x.startswith('https://api.')]
            if m:
                CF = m[-1]
                break
        except Exception:
            pass

if CF:
    open('CF_URL', 'w').write(CF)

KEY = globals().get('PANEL_KEY') or ''
full_url = (CF + '/?key=' + KEY) if (CF and KEY) else (CF or '')

print()
print('═' * 58)
if full_url:
    print('🌐 PANEL LINK (abhi khol sakte ho):')
    print('   ' + full_url)
    if CF and notify_publish(CF):
        print('📱 ntfy notification BHEJ DI — phone pe check karo! (tap = panel)')
    else:
        print('ℹ️ ntfy notification nahi gayi — uper wala link kholo')
else:
    print('⚠️ Link abhi nahi bana — 30 sec ruko, panel page / KEEPALIVE me khud dikhega')
print('═' * 58)
print('🔒 Panel ab KEY se locked hai — sirf uper wala FULL link chalega (bookmark karo)')
print('🎮 Minecraft: playit address waise hi — dost kuch nahi badalte')

# ── STAGE 10: SELF-HEALING AGENT (services + tunnels + backups + notifications) ──
import threading as _th

def alog(m):
    print('[' + datetime.now().strftime('%H:%M:%S') + '] ' + m, flush=True)

def _alive(pattern, exact=False):
    cmd = ['pgrep', '-x', pattern] if exact else ['pgrep', '-f', pattern]
    try:
        return subprocess.run(cmd, capture_output=True).returncode == 0
    except Exception:
        return False

def _srv_running():
    try:
        return 'mcserver' in subprocess.run(['screen', '-ls'], capture_output=True, text=True).stdout
    except Exception:
        return False

def _srv_ready():
    if not _srv_running():
        return False
    try:
        return 'Done (' in open('server_console.log', errors='ignore').read()[-4000:]
    except Exception:
        return False

def _stuff(c):
    try:
        subprocess.run(['screen', '-S', 'mcserver', '-X', 'stuff', c + '\n'], capture_output=True)
    except Exception:
        pass

def _min_jar():
    sw = ''
    try:
        sw = open('SOFTWARE').read().strip()
    except Exception:
        pass
    return 20 * 1024 if sw == 'fabric' else 40 * 1024 * 1024

def _jar_ok():
    try:
        return os.path.getsize('server.jar') > _min_jar()
    except Exception:
        return False

def _screen_kill():
    try:
        subprocess.run(['screen', '-S', 'mcserver', '-X', 'quit'], capture_output=True)
        subprocess.run(['screen', '-wipe'], capture_output=True)
    except Exception:
        pass

_ag = {'cf': None, 'ntfy_t': 0, 'online': None, 'ready': None, 'last_zip': 0, 'last_sync': 0, 'crash_t': 0}

def _ntfy(title, msg, click='', tags='video_game'):
    try:
        topic = globals().get('NTFY_TOPIC') or ''
        if not topic:
            return False
        h = {'Title': title, 'Tags': tags}
        if click:
            h['Click'] = click
        req = urllib.request.Request('https://ntfy.sh/' + topic, data=msg.encode(), headers=h)
        urllib.request.urlopen(req, timeout=10)
        return True
    except Exception:
        return False

def _start_server(why):
    if os.path.exists('STOP_FLAG') or _srv_running():
        return
    if not _jar_ok():
        alog('❌ server.jar nahi hai (' + why + ') — panel > Software me Apply karo ya notebook dobara Run karo')
        return
    try:
        if os.path.exists('START_LOCK') and time.time() - os.path.getmtime('START_LOCK') < 120:
            return
        open('START_LOCK', 'w').write(str(time.time()))
    except Exception:
        pass
    alog('🚀 Server start (' + why + ')')
    _screen_kill()
    subprocess.Popen(['screen', '-dmS', 'mcserver', 'bash', '-c',
                      JAVA + ' -Xmx2G -Xms2G -jar server.jar nogui 2>&1 | tee server_console.log'])
    try:
        open('SERVER_STARTED_AT', 'w').write(str(time.time()))
    except Exception:
        pass

def _ensure_services():
    if os.path.exists('panel.py') and not _alive('panel.py'):
        alog('🔄 Panel restart')
        subprocess.Popen(['python3', 'panel.py'], stdout=open('panel.log', 'w'), stderr=subprocess.STDOUT)
    if os.path.exists('cloudflared') and not _alive('cloudflared'):
        alog('🔄 cloudflared restart')
        subprocess.Popen(['./cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8081', '--no-autoupdate', '--protocol', 'http2'],
                         stdout=open('cloudflared.log', 'w'), stderr=subprocess.STDOUT)
    if PLAYIT_SECRET and os.path.exists('playit') and not _alive('playit', exact=True):
        alog('🔄 playit restart')
        subprocess.Popen(['./playit', '--secret', PLAYIT_SECRET, '--log-path', 'playit.log'],
                         stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    if not _srv_running():
        _start_server('watchdog')

def _url_tick():
    c = None
    try:
        if os.path.exists('cloudflared.log'):
            txt = open('cloudflared.log', errors='ignore').read()[-6000:]
            m = [x for x in re.findall(r'https://[a-z0-9-]+\.trycloudflare\.com', txt) if not x.startswith('https://api.')]
            c = m[-1] if m else None
    except Exception:
        c = None
    if c:
        try:
            open('CF_URL', 'w').write(c)
        except Exception:
            pass
        if _ag['cf'] != c:
            first = _ag['cf'] is None
            _ag['cf'] = c
            alog('🌐 Panel link: ' + c)
            if not first:
                full = c + (('/?key=' + PANEL_KEY) if globals().get('PANEL_KEY') else '')
                if _ntfy('MC Panel link', 'Panel ready: ' + full, full, 'link'):
                    alog('📱 ntfy notification bhej di')
    now = time.time()
    if c and now - _ag['ntfy_t'] > 3600:
        _ag['ntfy_t'] = now
        full = c + (('/?key=' + PANEL_KEY) if globals().get('PANEL_KEY') else '')
        _ntfy('MC Panel link (refresh)', 'Panel: ' + full, full, 'link')

def _url_monitor_loop():
    while True:
        try:
            _url_tick()
        except Exception:
            pass
        time.sleep(8)

def _watchdog():
    time.sleep(40)
    while True:
        try:
            _ensure_services()
            r = _srv_running()
            if r and _ag['online'] is not True:
                _ag['online'] = True
                alog('✅ Server process up')
            if (not r) and _ag['online'] is True:
                _ag['online'] = False
                _ag['ready'] = False
                if not os.path.exists('STOP_FLAG') and time.time() - _ag['crash_t'] > 300:
                    _ag['crash_t'] = time.time()
                    _ntfy('⚠️ Server restart ho raha hai', 'Crash/band hua — agent khud utha raha hai (30 sec)', '', 'warning')
            rdy = _srv_ready()
            if rdy and _ag['ready'] is not True:
                _ag['ready'] = True
                alog('🟢 Server READY (players join kar sakte hain)')
                _ntfy('🟢 Server ONLINE', 'Minecraft ready hai — khelo! 🎮')
            if (not rdy) and _ag['ready'] is True and not r:
                _ag['ready'] = False
        except Exception as e:
            alog('watchdog err: ' + str(e)[:120])
        time.sleep(30)

def _rsync_sources():
    srcs = [d for d in ('world', 'world_nether', 'world_the_end', 'plugins') if os.path.isdir(d)]
    files = [f for f in ('server.properties', 'ops.json', 'whitelist.json', 'banned-ips.json', 'banned-players.json', 'eula.txt') if os.path.isfile(f)]
    return srcs + files

def _sync_drive():
    if not DRIVE_BASE or not os.path.isdir(DRIVE_BASE):
        return False
    srcs = _rsync_sources()
    if not srcs:
        return False
    cmd = ['rsync', '-a', '--delete', '--exclude', '*.jar', '--exclude', 'session.lock', '--exclude', 'logs', '--exclude', 'cache', '--exclude', 'libraries'] + srcs + [DRIVE_BASE + '/']
    r = subprocess.run(cmd, capture_output=True)
    if r.returncode == 0:
        open('LAST_BACKUP', 'w').write(datetime.now().strftime('%H:%M:%S'))
        return True
    return False

def _make_zip(tag='backup'):
    try:
        if not DRIVE_BASE:
            return None
        bdir = os.path.join(DRIVE_BASE, 'backups')
        os.makedirs(bdir, exist_ok=True)
        name = tag + '-' + datetime.now().strftime('%Y%m%d-%H%M%S') + '.zip'
        args = ['zip', '-qr', os.path.join(bdir, name)] + _rsync_sources() + ['-x', '*.jar', '-x', 'session.lock', '-x', 'logs/*', '-x', 'cache/*', '-x', 'libraries/*']
        r = subprocess.run(args, capture_output=True)
        if r.returncode == 0 and os.path.exists(os.path.join(bdir, name)):
            zips = sorted([z for z in os.listdir(bdir) if z.endswith('.zip')], key=lambda z: os.path.getmtime(os.path.join(bdir, z)), reverse=True)
            for z in zips[12:]:
                try:
                    os.remove(os.path.join(bdir, z))
                except Exception:
                    pass
            return name
    except Exception as e:
        alog('zip err: ' + str(e)[:120])
    return None

def _do_restore(name):
    try:
        if not DRIVE_BASE:
            return 'Drive off hai'
        zp = os.path.join(DRIVE_BASE, 'backups', os.path.basename(name))
        if not os.path.exists(zp):
            return 'file nahi mili'
        open('RESTORE_STATUS', 'w').write('safety backup ban raha hai...')
        if _srv_running():
            _make_zip('pre-restore')
        open('STOP_FLAG', 'w').write('1')
        if _srv_running():
            open('RESTORE_STATUS', 'w').write('server band kar rahe hain (save)...')
            _stuff('save-all flush')
            time.sleep(3)
            _stuff('stop')
            for _ in range(25):
                time.sleep(3)
                if not _srv_running():
                    break
        _screen_kill()
        open('RESTORE_STATUS', 'w').write('restore ho raha hai: ' + name)
        tmp = '_restore_tmp'
        shutil.rmtree(tmp, ignore_errors=True)
        os.makedirs(tmp, exist_ok=True)
        r = subprocess.run(['unzip', '-qo', zp, '-d', tmp], capture_output=True)
        if r.returncode not in (0, 1):
            return 'unzip fail'
        for d in ('world', 'world_nether', 'world_the_end'):
            src = os.path.join(tmp, d)
            if os.path.isdir(src):
                shutil.rmtree(d, ignore_errors=True)
                shutil.move(src, d)
        if os.path.isdir(os.path.join(tmp, 'plugins')):
            os.makedirs('plugins', exist_ok=True)
            subprocess.run(['rsync', '-a', os.path.join(tmp, 'plugins') + '/', 'plugins/'], capture_output=True)
        for f in ('server.properties', 'ops.json', 'whitelist.json', 'banned-ips.json', 'banned-players.json'):
            src = os.path.join(tmp, f)
            if os.path.isfile(src):
                shutil.copy(src, f)
        shutil.rmtree(tmp, ignore_errors=True)
        open('RESTORE_STATUS', 'w').write('done ' + datetime.now().strftime('%H:%M:%S') + ' — server start ho raha hai')
        return 'ok'
    except Exception as e:
        return 'err: ' + str(e)[:120]
    finally:
        try:
            if os.path.exists('STOP_FLAG'):
                os.remove('STOP_FLAG')
        except Exception:
            pass

def _backup_loop():
    while True:
        time.sleep(5)
        try:
            if os.path.exists('RESTORE_REQUEST'):
                name = open('RESTORE_REQUEST').read().strip()
                try:
                    os.remove('RESTORE_REQUEST')
                except Exception:
                    pass
                res = _do_restore(name)
                alog('restore: ' + res)
                if res == 'ok':
                    _ntfy('🛟 Restore ho gaya', name + ' — server wapas aa raha hai', '', 'white_check_mark')
            elif os.path.exists('BACKUP_REQUEST'):
                try:
                    os.remove('BACKUP_REQUEST')
                except Exception:
                    pass
                open('BACKUP_STATUS', 'w').write('ban raha hai...')
                if _srv_running():
                    _stuff('save-all flush')
                    time.sleep(3)
                n = _make_zip('backup')
                open('BACKUP_STATUS', 'w').write(('done: ' + n) if n else 'fail (Drive?)')
                if n:
                    _ntfy('💾 Backup ban gaya', n, '', 'floppy_disk')
            else:
                now = time.time()
                if _srv_running() and DRIVE_BASE and now - _ag['last_sync'] > 60:
                    _ag['last_sync'] = now
                    _stuff('save-all')
                    time.sleep(3)
                    _sync_drive()
                if DRIVE_BASE and now - _ag['last_zip'] > 6 * 3600:
                    _ag['last_zip'] = now
                    _make_zip('auto')
        except Exception as e:
            alog('backup err: ' + str(e)[:120])

def _cmds_loop():
    time.sleep(15)
    while True:
        try:
            if _srv_running():
                _stuff('tps')
                time.sleep(2)
                _stuff('list')
        except Exception:
            pass
        time.sleep(55)

def _playit_check():
    try:
        if not os.path.exists('playit.log'):
            return
        txt = open('playit.log', errors='ignore').read()[-8000:]
        if txt.count('session expired') >= 2 or txt.count('SessionNotSetup') >= 2:
            alog('ℹ️ playit: purana session warning — normal hai, address same rahega')
    except Exception:
        pass

_playit_check()
_th.Thread(target=_watchdog, daemon=True).start()
_th.Thread(target=_url_monitor_loop, daemon=True).start()
_th.Thread(target=_backup_loop, daemon=True).start()
_th.Thread(target=_cmds_loop, daemon=True).start()
alog('🤖 Agent ON — services + 60sec auto-save/sync + backups + notifications')
_start_server('boot')

# 🔒 KEEPALIVE — Colab session ko jaagta rakho (fallback: JS click + busy loop)
# ⚠️ YE CELL HAMESHA CHALTA RAHEGA — ye normal hai! (Rokna ho to ■ Stop dabao)
import time, os
from datetime import datetime

try:
    from google.colab import output
    output.eval_js('''
    (function(){
      if (window.__mcKeepAlive) return "already-on";
      window.__mcKeepAlive = setInterval(function(){
        try {
          var btn = document.querySelector("colab-connect-button");
          if (btn) btn.click();
        } catch(e){}
      }, 60000);
      return "on";
    })();
    ''')
    print('🔒 Browser keep-alive ON')
except Exception as e:
    print('ℹ️ JS keepalive skip:', e)

print('🔒 Session keep-alive ON — Colab ko idle hone nahi dega. (Tab khula rakho!)')
n = 0
while True:
    time.sleep(300)
    n += 1
    line = '💚 Alive #' + str(n) + ' — ' + datetime.now().strftime('%H:%M:%S')
    if n % 3 == 0:
        try:
            cf = open('CF_URL').read().strip() if os.path.exists('CF_URL') else ''
            key = globals().get('PANEL_KEY') or ''
            full = (cf + '/?key=' + key) if (cf and key) else (cf or 'starting...')
            print(line + ' | 🌐 ' + full)
        except Exception:
            print(line)
    else:
        print(line)